In [ ]:
import os
import time
import pytesseract
import fitz
from pdf2image import convert_from_path
from PIL import Image
import re
import shutil
import pandas as pd
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
import requests
from pathlib import Path
from unidecode import unidecode
from datetime import datetime
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
# expression of interest notices

driver = webdriver.Chrome()
driver.get("LINK") #add link

pdf_blocks = driver.find_elements(By.CSS_SELECTOR, "div.file.pdf")

In [ ]:
tenders = []

file_blocks = driver.find_elements(By.CSS_SELECTOR, ".wpfd_list .file")

# create the folders
os.makedirs("saved_files", exist_ok=True)
os.makedirs("images", exist_ok=True)

today = datetime.now()

In [ ]:
# parse the dates

def extract_date(content_text):
    from datetime import datetime

    months_fr = {
        "janvier": "01",
        "fevrier": "02", "février": "02",
        "mars": "03",
        "avril": "04",
        "mai": "05",
        "juin": "06",
        "juillet": "07",
        "aout": "08", "août": "08",
        "septembre": "09",
        "octobre": "10",
        "novembre": "11",
        "decembre": "12", "décembre": "12"
    }

    content_text = unidecode(content_text.lower())
    content_text = re.sub(r'\s+', ' ', content_text)
    keyword_list = [
        "au plus tard le","au plus tard", "a ete reporte au", "a ete reporte le", 
        "a ete reporté au", "date limite de soumission de dossiers"
    ]

    for word in keyword_list:
        pos = content_text.find(word)
        if pos != -1:
            segment = content_text[pos:pos+60]
            match = re.search(r"(lundi|mardi|mercredi|jeudi|vendredi|samedi|dimanche)?\s*(\d{1,2})\s+(\w+)\s+(\d{4})", segment)
            if match:
                day, month_txt, year = match.group(2), match.group(3), match.group(4)
                month_num = months_fr.get(month_txt)
                if month_num:
                    return f"{day.zfill(2)}-{month_num}-{year}"
    return None


In [ ]:
# empty the saved_files folder
pdf_folder = "saved_files"
if os.path.exists(pdf_folder):
    shutil.rmtree(pdf_folder)
os.makedirs(pdf_folder, exist_ok=True)

# empty the saved_files folder
img_folder = "images"
if os.path.exists(img_folder):
    shutil.rmtree(img_folder)
os.makedirs(img_folder, exist_ok=True)

In [ ]:
# scraping

for i, block in enumerate(pdf_blocks[:10], 1):
    try:
        total_text = ""

        input_tag = block.find_element(By.CSS_SELECTOR, "input.wpfd_file_preview_link_download")
        link = input_tag.get_attribute("value")
        filename = os.path.join("saved_files", os.path.basename(link))

        if not os.path.exists(filename):
            r = requests.get(link, timeout=15)
            if r.status_code == 200 and "application/pdf" in r.headers.get("Content-Type", ""):
                with open(filename, "wb") as f:
                    f.write(r.content)
            else:
                continue
        else:
            continue

        doc = fitz.open(filename)

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            pix = page.get_pixmap(dpi=300)
            img_path = f"images/page_{i}_{page_num}.png"
            pix.save(img_path)

            try:
                with Image.open(img_path) as img:
                    text = pytesseract.image_to_string(img, lang="fra")
                    total_text += unidecode(text.lower()) + "\n"
            finally:
                try:
                    os.remove(img_path)
                except Exception as e:
                    continue

        doc.close()

        expiration_date = extract_date(total_text)
        if expiration_date:
            try:
                expiration_date_dt = datetime.strptime(expiration_date, "%d-%m-%Y")
                if expiration_date_dt > datetime.now():
                    try:
                        title = block.find_element(By.CSS_SELECTOR, "div.file-desc").text.strip()
                        publish_date = block.find_element(By.CSS_SELECTOR, "div.file-dated").text.strip()
                        publish_date = publish_date.replace("Date d'ajout:", "").strip()

                        tenders.append({
                            "title": title,
                            "publication_date": publish_date,
                            "expiration_date": expiration_date,
                            "link": link,
                            "country": "Guinea"
                        })
                    except Exception as e:
                        continue
                else:
                    continue
            except ValueError:
                continue
        else:
            continue

        try:
            os.remove(filename)
        except Exception as e:
            continue

    except Exception as e:
        print(f"Error in block {i}: {e}")

In [ ]:
# supplies

driver.get("https://armpguinee.org/appels-doffres/fourniture/")
pdf_blocks = driver.find_elements(By.CSS_SELECTOR, "div.file.pdf")

In [ ]:
# scraping

for i, block in enumerate(pdf_blocks[:10], 1):
    try:
        total_text = ""

        input_tag = block.find_element(By.CSS_SELECTOR, "input.wpfd_file_preview_link_download")
        link = input_tag.get_attribute("value")
        filename = os.path.join("saved_files", os.path.basename(link))

        if not os.path.exists(filename):
            r = requests.get(link, timeout=15)
            if r.status_code == 200 and "application/pdf" in r.headers.get("Content-Type", ""):
                with open(filename, "wb") as f:
                    f.write(r.content)
            else:
                continue
        else:
            continue

        doc = fitz.open(filename)

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            pix = page.get_pixmap(dpi=300)
            img_path = f"images/page_{i}_{page_num}.png"
            pix.save(img_path)

            try:
                with Image.open(img_path) as img:
                    text = pytesseract.image_to_string(img, lang="fra")
                    total_text += unidecode(text.lower()) + "\n"
            finally:
                try:
                    os.remove(img_path)
                except Exception as e:
                    continue
        doc.close()

        expiration_date = extract_date(total_text)
        if expiration_date:
            try:
                expiration_date_dt = datetime.strptime(expiration_date, "%d-%m-%Y")
                if expiration_date_dt > datetime.now():
                    try:
                        title = block.find_element(By.CSS_SELECTOR, "div.file-desc").text.strip()
                        publish_date = block.find_element(By.CSS_SELECTOR, "div.file-dated").text.strip()
                        publish_date = publish_date.replace("Date d'ajout:", "").strip()

                        tenders.append({
                            "title": title,
                            "publication_date": publish_date,
                            "expiration_date": expiration_date,
                            "link": link,
                            "country": "Guinea"
                        })
                    except Exception as e:
                        continue
                else:
                    continue
            except ValueError:
                continue
        else:
            continue

        try:
            os.remove(filename)
        except Exception as e:
            continue

    except Exception as e:
        print(f"Error in block {i}: {e}")

In [ ]:
driver.quit()

In [ ]:
df = pd.DataFrame(tenders)
df.to_excel("../2_scraped_files/Guinea2.xlsx", index=False) # modify